<a href="https://colab.research.google.com/github/chareeshnara04-max/Data-visualization-exercises_Chareesh_Naraa/blob/main/Who_Chooses_Us_Again%3F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# Load the data
df = pd.read_csv("aurora_air_survey.csv")

# Check the number of rows and columns
print("Shape:", df.shape)

# Create satisfied variable
df["satisfied"] = (df["csat"] >= 4).astype(int)

# Display first few rows
df.head()

Shape: (5000, 21)


,passenger_id,travel_type,class,flight,age,loyalty_member,arrival_delay_min,rating_wifi,rating_online_booking,rating_seat_comfort,...,rating_food,rating_entertainment,rating_staff,rating_boarding,rating_baggage,csat,nps_score,price_vs_competitor_pct,flew_again_12m,satisfied
0,100001,Business,Economy,Long-haul,31,Yes,6,5,5,4,...,5,5,4,5,4,5,10,-1,1,1
1,100002,Leisure,Economy,Short-haul,37,No,121,4,5,2,...,3,1,4,3,2,3,6,-11,0,0
2,100003,Leisure,Economy,Short-haul,35,No,0,4,5,5,...,4,4,5,5,4,5,10,-7,1,1
3,100004,Business,Economy,Long-haul,50,Yes,0,3,5,2,...,2,2,3,3,4,2,5,13,0,0
4,100005,Business,Business,Long-haul,60,Yes,0,4,3,3,...,3,5,5,4,5,5,10,4,1,1


In [2]:
print(df.shape)
print(df["satisfied"].value_counts())

(5000, 22)
satisfied
0    2700
1    2300
Name: count, dtype: int64


In [3]:
# Overall CSAT
overall_csat = (df["csat"] >= 4).mean() * 100

# CSAT by travel type
business_csat = (df.loc[df["travel_type"] == "Business", "csat"] >= 4).mean() * 100
leisure_csat = (df.loc[df["travel_type"] == "Leisure", "csat"] >= 4).mean() * 100

print(f"Overall CSAT: {overall_csat:.2f}%")
print(f"Business CSAT: {business_csat:.2f}%")
print(f"Leisure CSAT: {leisure_csat:.2f}%")

Overall CSAT: 46.00%
Business CSAT: 44.21%
Leisure CSAT: 47.16%


In [4]:
# Create NPS groups
df["nps_group"] = pd.cut(
    df["nps_score"],
    bins=[-1, 6, 8, 10],
    labels=["Detractor", "Passive", "Promoter"]
)

# Function to calculate NPS
def calculate_nps(data):
    promoters = (data["nps_score"] >= 9).mean() * 100
    detractors = (data["nps_score"] <= 6).mean() * 100
    return promoters - detractors

# Calculate NPS
overall_nps = calculate_nps(df)

business_nps = calculate_nps(
    df[df["travel_type"] == "Business"]
)

leisure_nps = calculate_nps(
    df[df["travel_type"] == "Leisure"]
)

print(f"Overall NPS: {overall_nps:.2f}")
print(f"Business NPS: {business_nps:.2f}")
print(f"Leisure NPS: {leisure_nps:.2f}")

# Show group counts
print("\nOverall NPS groups:")
print(df["nps_group"].value_counts())

Overall NPS: -4.04
Business NPS: -8.59
Leisure NPS: -1.09

Overall NPS groups:
nps_group
Passive      1918
Detractor    1642
Promoter     1440
Name: count, dtype: int64


Business travellers are less happy, because their NPS (-8.59) is lower than Leisure (-1.09).

Also, the overall NPS is negative, meaning there are more detractors than promoters.

Your NPS group counts are:

Promoters: 1,440
Passives: 1,918
Detractors: 1,642

In [6]:
# Share who flew with Aurora again within 12 months
return_by_nps = df.groupby("nps_group")["flew_again_12m"].mean() * 100

print(return_by_nps)

nps_group
Detractor    25.578563
Passive      50.938478
Promoter     67.013889
Name: flew_again_12m, dtype: float64


/tmp/ipykernel_3429/3847055500.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return_by_nps = df.groupby("nps_group")["flew_again_12m"].mean() * 100


In [7]:
promoter_return = df.loc[df["nps_group"] == "Promoter", "flew_again_12m"].mean()
detractor_return = df.loc[df["nps_group"] == "Detractor", "flew_again_12m"].mean()

times_more_likely = promoter_return / detractor_return

print(f"Promoter return rate: {promoter_return * 100:.2f}%")
print(f"Detractor return rate: {detractor_return * 100:.2f}%")
print(f"Promoters are {times_more_likely:.2f}x more likely to come back.")

Promoter return rate: 67.01%
Detractor return rate: 25.58%
Promoters are 2.62x more likely to come back.


In [8]:
from sklearn.linear_model import LogisticRegression
import numpy as np

# Prepare the inputs
X = pd.DataFrame({
    "nps_score": df["nps_score"],
    "loyalty_member": (df["loyalty_member"] == "Yes").astype(int),
    "price_vs_competitor": df["price_vs_competitor_pct"] / 10,
    "arrival_delay": df["arrival_delay_min"] / 30
})

# Outcome
y = df["flew_again_12m"]

# Fit logistic regression
model = LogisticRegression(max_iter=1000)
model.fit(X, y)

# Convert coefficients to odds multipliers
odds_multipliers = np.exp(model.coef_[0])

for variable, multiplier in zip(X.columns, odds_multipliers):
    print(f"{variable}: {multiplier:.3f}")

nps_score: 1.542
loyalty_member: 1.778
price_vs_competitor: 0.731
arrival_delay: 0.858


A 1-point increase in NPS is associated with 1.542× the odds of flying again, holding the other variables constant.